# Data Collection & Preparation

**Dataset choice: SQuAD v1.1 (Stanford Question Answering Dataset)**

Why SQuAD?
- Real questions written by crowd workers about Wikipedia articles
- Each question is paired with the exact Wikipedia paragraph that contains the answer
- Paragraphs are 100–250 words — natural document chunks, no manual splitting needed
- `rajpurkar/squad` is reliably available on HuggingFace with no auth required
- ~87,000 (question, paragraph) pairs — enough for contrastive training

**What is query / document here?**
- Query: a natural language question (e.g. "What river does the Colorado River flow into?")
- Document: a Wikipedia paragraph that contains the answer

**Pipeline:**
1. Download SQuAD v1.1 via `rajpurkar/squad`
2. Deduplicate contexts (same paragraph appears for multiple questions)
3. Assign each unique paragraph a stable `doc_id`
4. Build triplets: (query, positive_paragraph, negative_paragraph)
5. Split 80/10/10 train/val/test — no query overlap between splits
6. Save to `data/processed/`


In [ ]:
import sys
print(sys.executable)
print(sys.version)

/Library/Developer/CommandLineTools/usr/bin/python3
3.9.6 (default, Apr 17 2026, 18:15:52) 
[Clang 21.0.0 (clang-2100.1.1.101)]


In [1]:
import os
import json
import random
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

Directories ready.


## 1. Download SQuAD v1.1 via HuggingFace datasets

In [ ]:
from datasets import load_dataset

# SQuAD v1.1 — question answering over Wikipedia paragraphs
print('Loading SQuAD v1.1 train split...')
dataset = load_dataset('rajpurkar/squad', split='train')
print(f'Train size: {len(dataset):,} examples')
print(f'Columns: {dataset.column_names}')
print()
print('Example:')
ex = dataset[0]
print(f'  question: {ex["question"]}')
print(f'  context:  {ex["context"][:200]}...')
print(f'  answer:   {ex["answers"]["text"][0]}')

Loading SQuAD v1.1 train split...


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Train size: 87,599 examples
Columns: ['id', 'title', 'context', 'question', 'answers']

Example:
  question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  context:  Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
  answer:   Saint Bernadette Soubirous


## 2. Extract (query, positive_passage) pairs

SQuAD structure per row:
- `question`: the natural language question → this is our **query**
- `context`: the Wikipedia paragraph containing the answer → this is our **positive document**
- `title`: article title (useful for grouping)
- `answers`: the answer span (we don't use this — we only need question + context)

Each unique `context` becomes one document in our corpus. Multiple questions can share the same context paragraph.

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())


def chunk_text(text: str, min_words: int = 50, max_words: int = 300) -> list[str]:
    """Split text into chunks of min_words–max_words using sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_wc = [], [], 0
    for sent in sentences:
        wc = len(sent.split())
        if current_wc + wc > max_words and current:
            chunks.append(' '.join(current))
            current, current_wc = [], 0
        current.append(sent)
        current_wc += wc
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if word_count(c) >= min_words]


print('chunk_text test:')
sample = ' '.join(['This is sentence number %d.' % i for i in range(60)])
chunks = chunk_text(sample)
print(f'  Input words: {word_count(sample)}, Chunks: {len(chunks)}, Sizes: {[word_count(c) for c in chunks]}')

In [ ]:
# SQuAD train has ~87k rows — we use all of them
pairs = []   # list of {'query': str, 'positive': str}

# Build corpus: deduplicate contexts first
context_to_id = {}
corpus = {}       # doc_id -> text

print('Building corpus and (query, positive_doc) pairs...')
for row in tqdm(dataset, total=len(dataset)):
    query   = row['question'].strip()
    context = row['context'].strip()

    # Register context in corpus (dedup by exact text)
    if context not in context_to_id:
        doc_id = f'doc_{len(context_to_id)}'
        context_to_id[context] = doc_id
        # Chunk if the paragraph is longer than 300 words
        chunks = chunk_text(context) if word_count(context) > 300 else [context]
        # Use the first chunk as the canonical document
        corpus[doc_id] = chunks[0]

    pairs.append({
        'query':       query,
        'positive_id': context_to_id[context],
    })

print(f'\nTotal pairs:      {len(pairs):,}')
print(f'Unique documents: {len(corpus):,}')
print(f'\nExample pair:')
p = pairs[0]
print(f'  QUERY:    {p["query"]}')
print(f'  POSITIVE: {corpus[p["positive_id"]][:200]}...')

## 3. Corpus summary

The corpus is already built: each unique Wikipedia paragraph has a stable `doc_id`. This is the pool the retrieval system will index.

In [ ]:
corpus_ids = list(corpus.keys())
print(f'Corpus size:  {len(corpus_ids):,} unique passages')
print(f'Total pairs:  {len(pairs):,}')
print(f'Avg doc length: {sum(word_count(t) for t in corpus.values()) / len(corpus):.1f} words')

## 4. Build Triplets (query, positive, negative)

**Negative sampling strategy:**
- **Easy negatives**: randomly sample a passage from the corpus that is NOT the positive.
  - Simple, fast, sufficient for initial training.
- **Hard negatives** (optional, better): passages that are topically similar but not the answer.
  - We add 1 hard negative per triplet using BM25 once the index is built.
  - For now, we use in-batch negatives during training (InfoNCE handles this automatically).

For the explicit triplet file (used for evaluation), we sample 1 random negative per query.

In [ ]:
triplets = []
for p in tqdm(pairs, desc='Building triplets'):
    pos_id = p['positive_id']
    # Sample a random negative (not the positive)
    neg_id = pos_id
    while neg_id == pos_id:
        neg_id = random.choice(corpus_ids)
    triplets.append({
        'query':       p['query'],
        'positive_id': pos_id,
        'negative_id': neg_id,
    })

print(f'Triplets built: {len(triplets):,}')

## 5. Train / Validation / Test Split (80 / 10 / 10)

Shuffle first, then split. We ensure **no query appears in both train and test**.

In [ ]:
random.shuffle(triplets)

n = len(triplets)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_triplets = triplets[:n_train]
val_triplets   = triplets[n_train : n_train + n_val]
test_triplets  = triplets[n_train + n_val :]

# Verify no query overlap between train and test
train_queries = {t['query'] for t in train_triplets}
test_queries  = {t['query'] for t in test_triplets}
overlap = train_queries & test_queries
print(f'Train: {len(train_triplets):,}  |  Val: {len(val_triplets):,}  |  Test: {len(test_triplets):,}')
print(f'Query overlap between train and test: {len(overlap)} (should be 0 or minimal)')

## 6. Save to disk

In [ ]:
def save_jsonl(data: list, path: Path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Saved {len(data):,} records → {path}')


# Corpus (passage pool)
with open(PROCESSED_DIR / 'corpus.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print(f'Saved corpus ({len(corpus):,} passages) → {PROCESSED_DIR / "corpus.json"}')

# Triplet splits
save_jsonl(train_triplets, PROCESSED_DIR / 'train.jsonl')
save_jsonl(val_triplets,   PROCESSED_DIR / 'val.jsonl')
save_jsonl(test_triplets,  PROCESSED_DIR / 'test.jsonl')

## 7. Dataset Analysis

In [ ]:
import matplotlib.pyplot as plt

# --- Length statistics ---
query_lengths = [word_count(t['query']) for t in triplets]
doc_lengths   = [word_count(corpus[t['positive_id']]) for t in triplets]

print('=== Query length (words) ===')
print(f'  Mean:   {np.mean(query_lengths):.1f}')
print(f'  Median: {np.median(query_lengths):.1f}')
print(f'  Min:    {np.min(query_lengths)}')
print(f'  Max:    {np.max(query_lengths)}')

print()
print('=== Positive passage length (words) ===')
print(f'  Mean:   {np.mean(doc_lengths):.1f}')
print(f'  Median: {np.median(doc_lengths):.1f}')
print(f'  Min:    {np.min(doc_lengths)}')
print(f'  Max:    {np.max(doc_lengths)}')

print()
print(f'Total triplets:      {len(triplets):,}')
print(f'Unique passages:     {len(corpus):,}')

# --- Histogram ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(query_lengths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Query Length Distribution (words)')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Count')

axes[1].hist(doc_lengths, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Positive Passage Length Distribution (words)')
axes[1].set_xlabel('Word count')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'length_distributions.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# --- Print 5 example triplets ---
print('=== 5 Example Triplets ===\n')
for i, t in enumerate(test_triplets[:5]):
    pos_text = corpus[t['positive_id']]
    neg_text = corpus[t['negative_id']]
    print(f'--- Example {i+1} ---')
    print(f'QUERY:    {t["query"]}')
    print(f'POSITIVE: {pos_text[:200]}...' if len(pos_text) > 200 else f'POSITIVE: {pos_text}')
    print(f'NEGATIVE: {neg_text[:200]}...' if len(neg_text) > 200 else f'NEGATIVE: {neg_text}')
    print()

## Summary

| Item | Value |
|------|-------|
| Dataset | SQuAD v1.1 (`rajpurkar/squad`) |
| Query type | Natural language questions about Wikipedia articles |
| Document type | Wikipedia paragraphs (100–250 words) |
| Triplets total | ~87,000 |
| Train / Val / Test | 80% / 10% / 10% |
| Negative type | Random in-corpus negatives (easy); in-batch negatives used during InfoNCE training |

**Why this data is suitable:**
- Questions are written by humans about real Wikipedia text — not synthetic
- Low lexical overlap between question and paragraph forces semantic understanding
- Paragraphs are self-contained and naturally sized (no aggressive chunking needed)
- Scale (~87k pairs) is sufficient to train a contrastive bi-encoder in a few epochs

**Next step → Baselines (BM25 + TF-IDF)**